# NumPy

Has visto las funciones de Python *vainilla* (sin librerías desarrolladas por terceros).
En particular, del módulo `math` y que operan principalmente sobre números individuales.
Python no es un lenguaje diseñado para trabajo numérico. Por eso la manipulación de
arreglos de números ha sido tan incómoda (si no crees que fue incómoda, lo creeras
después de avanzar más en esta guía).

Python es uno de los lenguajes más usados por la comunidad científica
[@Virtanen2020], pero nadie usa Python como lo hemos hecho en los ejemplos
anteriores. Esencialmente todas las bibliotecas comunes en trabajo científico
funcionan gracias a NumPy [-@Harris2020].

## ¿Qué es?

El equipo de NumPy resume su trabajo muy bien.

> NumPy es el paquete fundamental para la computación científica en Python.
> Es una biblioteca de Python que proporciona un objeto de matriz multidimensional,
> diversos objetos derivados y una variedad de rutinas para operaciones rápidas en matrices,
> que incluyen operaciones matemáticas, lógicas, manipulación de formas, ordenamiento,
> selección, E/S, transformadas discretas de Fourier,
> álgebra lineal básica, operaciones estadísticas básicas, simulación aleatoria y mucho más.
>
> -- [Documentación de NumPy](https://numpy.org/doc/2.5/user/whatisnumpy.html)

NumPy incorpora arreglos numéricos a Python, e implementa métodos para
utilizarlos y modificarlos de forma rápida y eficiente.

## ¿Cómo funciona?

Las herramientas numéricas de NumPy no están escritas en Python.
Están escritas (principlamente) en C, un lenguaje compilado
(a diferencia de Python, que es interpretado)
y de más bajo nivel
(_i.e._ que ofrece menos abstracciones y conveniencias).
Al compilar código de C se generan instrucciones directamente para el procesador de
la computadora sobre qué hacer con los números y cómo. Esto permite optimizar
las instrucciones y aprovechar la verdadera velocidad que las computadoras
tienen para manipular números y arreglos de números.

Python, en cambio, ofrece estructuras que no requieren saber cómo se maneja
la memoria en la computadora. Como el intérprete de Python debe estar preparado para
manejar cualquier tipo de número o estructura y hacer lo que le pidas con ella en
cualquier momento, no puede optimizar estas instrucciones.

NumPy manipula arreglos numéricos usando un _back-end_ escrito
en C y optimizado. Al importar NumPy a un _script_ de Python, nuestros datos
pueden ser enviados a esta biblioteca de C y operados con estas optimizaciones.

In [ ]:
import numpy as np

Para usar estas optimizaciones, NumPy guarda estos datos en nuevo tipo llamado
`ndarray`. Es, a primera vista, muy similar a las listas en Python.

In [ ]:
a1D = np.array([1, 2, 3, 4])
print(a1D)
print(a1D[0], a1D[-1])
print(a1D[1:3])

Sin embargo, tanto la estructura completa como sus elementos son tipos diferentes a los
de Python puro.

In [ ]:
print(type(a1D))
print(type(a1D[0]))

No tienes que preocuparte de los detalles técnicos. Sólo se consciente de que tus
números ya no están en tuplas `()` o listas `[]`, sino en una estructura nueva.
Tampoco tienes que preocuparte por fijar estos tipos manualmente; NumPy lo hace
automáticamente.

### Álgebra de arreglos

Estos objetos se comportan como esperaríamos de vectores matemáticos o matrices.
El operador `+` suma elemento a elemento (en lugar de concatenar, como lo harían las
lsitas de Python).

In [ ]:
b1D = np.array([4, 3, 2, 1])
print(a1D + b1D)

La multiplicación por escalar se aplica a cada elemento del arreglo.

In [ ]:
k = 5.5
print(k * b1D)
print(type((k * b1D)[0]))

(NumPy convierte automáticamente a `k` de un `float` a un `numpy.float`, y convierte
el arreglo de `numpy.int` a `numpy.float`.)

Similarmente se pueden crear arreglos de 2 dimensiones (como matrices) o de dimensiones
arbitrarias. Las operaciones aritméticas se hacen elemento a elemento.

In [ ]:
a2D = np.array([[1, 2], [3, 4], [5, 6]])
print(a2D)

In [ ]:
b2D = np.array([[2, 3], [4, 5], [6, 7]])
print(b2D)

In [ ]:
c2D = a2D + b2D
print(c2D)

In [ ]:
print(a2D * 2)
print(b2D * 0.5)

In [ ]:
a3D = np.array(
    [
        [[1, 2, 3], [4, 5, 6], [7, 8, 9]],
        [[10, 11, 12], [13, 14, 15], [16, 17, 18]],
        [[19, 20, 21], [22, 23, 24], [25, 26, 27]],
    ]
)
# Tal vez es un poco complicado visualizar un cubo de 3x3x3 números, pero eso es lo que
# genera este código

print(a3D[0, 1, 0], a3D[0, 2, 2], a3D[2, 2, 0])

Habrás notado que, a diferencia de las listas de Python, podemos accesar elementos
de la forma más natural `[i, j, k]` en lugar de `[i][j][k]`.

### Vectorización

NumPy ofrece implementaciones de *casi* todas las funciones matemáticas del módulo
`math`. Sin embargo, NumPy permite aplicarlas a arreglos de números de forma más sencilla.

Calculemos la raíz cuadrada de cada uno de los números en el cubo de $3 \times 3 \times 3$
que definimos arriba (preservando la estructura del arreglo).
Sin NumPy, habríamos de hacer algo como
```python
roots = [
    [[math.sqrt(a3D[i][j][k]) for k in range(3)] for j in range(3)] for i in range(3)
]
```
Con NumPy, en cambio…

In [ ]:
roots = np.sqrt(a3D)
print(roots)
print(roots[0, 1, 0], roots[0, 2, 2], roots[2, 2, 0])

Ni siquiera es necesario especificar las dimensiones del arreglo. NumPy se encarga
de todo.
Hasta se encarga de que los números esten alineados al usar `print`.
No solo es más rápido de escribir, la ejecución es mas rápida también.

In [ ]:
import math

In [ ]:
long_array = [l for l in range(5000000)]

In [ ]:
%timeit -r 7 -n 10 [math.sqrt(x) for x in long_array]

In [ ]:
%timeit -r 7 -n 10 np.sqrt(long_array)

Esto se debe a que el _back-end_ de NumPy automáticamente paraleliza (vectoriza) estas operaciones,
permitiendo que se realizen varias a la vez en lugar de una por una.

::: {hint} Pruebas de rendimiento (_benchmarking_)
Jupyter (a través de IPython) ofrece algunas herramientas simples para evaluar
el rendimiento de algoritmos y comandos.
El macro `%time` mide el tiempo de evaluación de un comando y
`%timeit` lo ejecuta varias veces para medir tiempo promedio y dispersión.

Las variantes con `%%` en lugar de `%` funcionan en toda la celda, en lugar de solo la
linea donde se coloque. Debes colocar estas variantes al principio.
Nota que lo que siga inmediatamente al macro no se mide, se considera "preparación"
para el resto de la celda.
:::

Otras operaciones comunes también se vectorizan.

In [ ]:
np.array([[1, 2, 3], [4, 5, 6]]) / 6

In [ ]:
np.array([[1, 2, 3], [4, 5, 6]]) ** 3